In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "banco"
SCHEMA = "banking_analytics"

CUSTOMER_SOURCE = f"{CATALOG}.{SCHEMA}.silver_customers"
CUSTOMER_TARGET = f"{CATALOG}.{SCHEMA}.dim_customer"

# Leer tablas
df_source_customer = spark.table(CUSTOMER_SOURCE)
df_target_customer = spark.table(CUSTOMER_TARGET)

print("===== SILVER CUSTOMERS =====")
df_source_customer.printSchema()

print("===== DIM CUSTOMER =====")
df_target_customer.printSchema()

print("===== CONTEOS =====")
print("Registros Silver:", df_source_customer.count())
print("Registros Gold:", df_target_customer.count())

In [0]:
# Hacer una prueba
from pyspark.sql.window import Window

# 1. Seleccionar dos clientes existentes para simular actualizaciones
df_base = (
    df_source_customer
    .orderBy("customer_id")
    .limit(2)
    .withColumn(
        "_rn",
        F.row_number().over(Window.orderBy("customer_id"))
    )
)

# 2. Simular modificaciones de clientes existentes
df_updates = (
    df_base
    .withColumn(
        "customer_name",
        F.concat(F.col("customer_name"), F.lit(" - Actualizado"))
    )
    .withColumn(
        "email",
        F.concat(
            F.lit("updated_"),
            F.col("customer_id"),
            F.lit("@example.com")
        )
    )
    .withColumn("customer_segment", F.lit("PREMIUM"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
    .withColumn("_source_file", F.lit("incremental_customers_20261118.csv"))
    .withColumn("_operation", F.lit("UPDATE"))
    .drop("_rn")
)

# 3. Simular dos clientes nuevos
df_inserts = (
    df_base
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("NEW_CUST_20261118_"),
            F.lpad(F.col("_rn").cast("string"), 3, "0")
        )
    )
    .withColumn(
        "customer_name",
        F.format_string("Cliente Nuevo %03d", F.col("_rn"))
    )
    .withColumn(
        "email",
        F.concat(
            F.lit("new_customer_"),
            F.col("_rn"),
            F.lit("@example.com")
        )
    )
    .withColumn("registration_date", F.current_date())
    .withColumn("customer_segment", F.lit("STANDARD"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
    .withColumn("_source_file", F.lit("incremental_customers_20261118.csv"))
    .withColumn("_operation", F.lit("INSERT"))
    .drop("_rn")
)

# 4. Unificar el lote incremental
df_batch_customers = df_updates.unionByName(df_inserts)

# 5. Mostrar el lote
display(
    df_batch_customers.select(
        "customer_id",
        "customer_name",
        "email",
        "customer_segment",
        "_operation"
    ).orderBy("_operation", "customer_id")
)

In [0]:
print("Total registros del lote:", df_batch_customers.count())

display(
    df_batch_customers
    .groupBy("_operation")
    .count()
)

# Verificar que los IDs nuevos no existan en Gold
df_new_ids = (
    df_batch_customers
    .filter(F.col("_operation") == "INSERT")
    .select("customer_id")
)

df_existing_ids = (
    df_target_customer
    .select("customer_id")
)

new_id_conflicts = (
    df_new_ids
    .join(df_existing_ids, "customer_id", "inner")
    .count()
)

print("Conflictos de IDs nuevos:", new_id_conflicts)

# Verificar unicidad dentro del lote
duplicate_ids = (
    df_batch_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("IDs duplicados en el lote:", duplicate_ids)

In [0]:
# Preparar los datos para el MERGE
# Preparar registros para la dimensión de clientes
df_customer_merge = (
    df_batch_customers
    .withColumn(
        "customer_key",
        F.xxhash64("customer_id")
    )
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "birth_date",
        "country",
        "registration_date",
        "email",
        "customer_segment",
        "_operation"
    )
)

display(df_customer_merge)

In [0]:
print("Total registros preparados:", df_customer_merge.count())

print(
    "Claves sustitutas nulas:",
    df_customer_merge.filter(
        F.col("customer_key").isNull()
    ).count()
)

print(
    "Claves sustitutas duplicadas:",
    df_customer_merge.groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    df_customer_merge
    .groupBy("_operation")
    .count()
)

In [0]:
# Referencia a la tabla Delta destino
delta_customer = DeltaTable.forName(
    spark,
    CUSTOMER_TARGET
)

# Ejecutar MERGE
(
    delta_customer.alias("target")
    .merge(
        df_customer_merge.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_key": "source.customer_key",
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .execute()
)

In [0]:
# Validar resultados
df_customer_after_merge = spark.table(CUSTOMER_TARGET)

print(
    "Clientes antes del MERGE:",
    df_target_customer.count()
)

print(
    "Clientes después del MERGE:",
    df_customer_after_merge.count()
)

print(
    "Clientes nuevos insertados:",
    df_customer_after_merge.filter(
        F.col("customer_id").startswith("NEW_CUST_20261118_")
    ).count()
)

display(
    df_customer_after_merge
    .filter(
        F.col("customer_id").startswith("NEW_CUST_20261118_")
    )
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "customer_segment"
    )
)

In [0]:
# Validar integridad de datos
# 1. Verificar los clientes que fueron actualizados
df_updated_check = (
    df_customer_after_merge
    .filter(F.col("customer_name").contains(" - Actualizado"))
)

print("Clientes actualizados:", df_updated_check.count())

display(
    df_updated_check.select(
        "customer_key",
        "customer_id",
        "customer_name",
        "email",
        "customer_segment"
    )
)

# 2. Verificar unicidad de customer_id
duplicate_customer_ids = (
    df_customer_after_merge
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Customer IDs duplicados:", duplicate_customer_ids)

# 3. Verificar unicidad de customer_key
duplicate_customer_keys = (
    df_customer_after_merge
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Customer keys duplicadas:", duplicate_customer_keys)

# 4. Verificar claves sustitutas nulas
null_customer_keys = (
    df_customer_after_merge
    .filter(F.col("customer_key").isNull())
    .count()
)

print("Customer keys nulas:", null_customer_keys)

In [0]:
# Probar la indepotencia
delta_customer = DeltaTable.forName(
    spark,
    CUSTOMER_TARGET
)

(
    delta_customer.alias("target")
    .merge(
        df_customer_merge.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_key": "source.customer_key",
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .execute()
)

In [0]:
df_customer_idempotency = spark.table(CUSTOMER_TARGET)

print("Clientes después del segundo MERGE:",
      df_customer_idempotency.count())

print(
    "Clientes nuevos:",
    df_customer_idempotency.filter(
        F.col("customer_id").startswith("NEW_CUST_20261118_")
    ).count()
)

print(
    "Customer IDs duplicados:",
    df_customer_idempotency
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
# Auditar el historial
# Consultar historial de la dimensión
df_customer_history = spark.sql(
    f"DESCRIBE HISTORY {CUSTOMER_TARGET}"
)

display(
    df_customer_history.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    ).orderBy(F.col("version").desc())
)

In [0]:
# Evitar actualizaciones innecesarias
delta_customer = DeltaTable.forName(
    spark,
    CUSTOMER_TARGET
)

# Detectar cambios en los atributos del cliente
update_condition = """
NOT (
    target.customer_name <=> source.customer_name
    AND target.birth_date <=> source.birth_date
    AND target.country <=> source.country
    AND target.registration_date <=> source.registration_date
    AND target.email <=> source.email
    AND target.customer_segment <=> source.customer_segment
)
"""

(
    delta_customer.alias("target")
    .merge(
        df_customer_merge.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        condition=update_condition,
        set={
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_key": "source.customer_key",
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "birth_date": "source.birth_date",
            "country": "source.country",
            "registration_date": "source.registration_date",
            "email": "source.email",
            "customer_segment": "source.customer_segment"
        }
    )
    .execute()
)

In [0]:
# Revisar las metricas
df_history = spark.sql(
    f"DESCRIBE HISTORY {CUSTOMER_TARGET}"
)

display(
    df_history
    .filter(F.col("operation") == "MERGE")
    .select(
        "version",
        "timestamp",
        "operationMetrics"
    )
    .orderBy(F.col("version").desc())
    .limit(3)
)

In [0]:
df_customer_final = spark.table(CUSTOMER_TARGET)

df_customer_final.select(
    F.count("*").alias("total_customers"),
    F.countDistinct("customer_id").alias("unique_customer_ids"),
    F.countDistinct("customer_key").alias("unique_customer_keys"),
    F.sum(
        F.when(F.col("customer_key").isNull(), 1).otherwise(0)
    ).alias("null_customer_keys")
).display()

In [0]:
# Inspeccionar las tablas bronze
BRONZE_TRANSACTIONS = f"{CATALOG}.{SCHEMA}.bronze_transactions"

df_bronze_transactions = spark.table(BRONZE_TRANSACTIONS)

display(df_bronze_transactions.limit(5))

In [0]:
## Simular retrazo de transacciones
SILVER_ACCOUNTS = f"{CATALOG}.{SCHEMA}.silver_accounts"

df_valid_account = (
    spark.table(SILVER_ACCOUNTS)
    .select("account_id")
    .orderBy("account_id")
    .limit(1)
)

display(df_valid_account)

In [0]:
from pyspark.sql import Row
from datetime import datetime, date

late_transaction = [(
    "ACC000141",
    850.50,
    "MOBILE_APP",
    "PEN",
    "Late arriving transaction",
    "COMPLETED",
    "2026-11-10T14:30:00",
    "TRX_LATE_000001",
    "PAYMENT",
    datetime(2026, 11, 19, 18, 0, 0),
    "late_transactions.jsonl",
    date(2026, 11, 19)
)]

df_late_transaction = spark.createDataFrame(
    late_transaction,
    schema=df_bronze_transactions.schema
)

display(df_late_transaction)

In [0]:
# Simulamos el último watermark procesado
last_watermark = "2026-11-18T23:59:59"

# Simulamos la llegada del nuevo registro a Bronze
df_bronze_with_late = (
    df_bronze_transactions
    .unionByName(df_late_transaction)
)

# Detectamos los registros posteriores al watermark
df_new_records = (
    df_bronze_with_late
    .filter(
        F.col("_ingestion_timestamp") >
        F.to_timestamp(F.lit(last_watermark))
    )
)

display(df_new_records)

In [0]:
# Realizar validacion de la transaccion
SILVER_TRANSACTIONS = f"{CATALOG}.{SCHEMA}.silver_transactions"

df_silver_transactions = spark.table(SILVER_TRANSACTIONS)

# Validar si la transacción ya existe
existing_transaction = (
    df_silver_transactions
    .filter(F.col("transaction_id") == "TRX_LATE_000001")
    .count()
)

# Validar si la cuenta existe
valid_account = (
    spark.table(SILVER_ACCOUNTS)
    .filter(F.col("account_id") == "ACC000141")
    .count()
)

# Validar condiciones básicas
basic_validation = (
    df_late_transaction
    .filter(
        F.col("transaction_id").isNotNull()
        & (F.col("transaction_id") != "")
        & F.col("account_id").isNotNull()
        & (F.col("amount") > 0)
        & F.col("currency").isin("PEN", "USD")
        & F.col("status").isin("COMPLETED", "PENDING", "REJECTED")
        & F.col("transaction_type").isin(
            "DEPOSIT", "PAYMENT", "TRANSFER", "WITHDRAWAL"
        )
    )
    .count()
)

print(f"Transacciones existentes en Silver: {existing_transaction}")
print(f"Coincidencias de cuenta en Silver: {valid_account}")
print(f"Registros que pasan validaciones básicas: {basic_validation}")

In [0]:
df_silver_transactions.printSchema()

In [0]:
# Darle foma de tabla silver transacciones
df_late_silver = (
    df_late_transaction
    .select(
        F.col("account_id").cast("string"),
        F.col("amount").cast("decimal(18,2)"),
        F.col("channel").cast("string"),
        F.col("currency").cast("string"),
        F.col("description").cast("string"),
        F.col("status").cast("string"),
        F.to_timestamp("transaction_date").alias("transaction_date"),
        F.col("transaction_id").cast("string"),
        F.col("transaction_type").cast("string"),
        F.col("_ingestion_timestamp").cast("timestamp"),
        F.col("_source_file").cast("string"),
        F.to_date("_ingestion_date").alias("_ingestion_date")
    )
)

display(df_late_silver)

In [0]:
# Comparar los esquemas
schemas_match = (
    df_late_silver.schema == df_silver_transactions.schema
)

print(f"¿Los esquemas coinciden?: {schemas_match}")
print(f"Columnas origen: {len(df_late_silver.columns)}")
print(f"Columnas destino: {len(df_silver_transactions.columns)}")
print(f"Registros preparados: {df_late_silver.count()}")

In [0]:
# Ingresar la transaction tardia
delta_transactions = DeltaTable.forName(
    spark,
    SILVER_TRANSACTIONS
)

(
    delta_transactions.alias("target")
    .merge(
        df_late_silver.alias("source"),
        "target.transaction_id = source.transaction_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
# Auditar la transaccion
# Verificar la transacción insertada
df_inserted = (
    spark.table(SILVER_TRANSACTIONS)
    .filter(F.col("transaction_id") == "TRX_LATE_000001")
)

# Auditoría
df_inserted.select(
    F.count("*").alias("transaction_count"),
    F.countDistinct("transaction_id").alias("unique_transaction_ids"),
    F.sum("amount").alias("transaction_amount")
).display()

In [0]:
# Consultar el historial de Delta
df_transaction_history = spark.sql(
    f"DESCRIBE HISTORY {SILVER_TRANSACTIONS}"
)

display(
    df_transaction_history
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
    .limit(3)
)

In [0]:
# Probando idempotencia
delta_transactions = DeltaTable.forName(
    spark,
    SILVER_TRANSACTIONS
)

(
    delta_transactions.alias("target")
    .merge(
        df_late_silver.alias("source"),
        "target.transaction_id = source.transaction_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_final_audit = spark.table(SILVER_TRANSACTIONS)

df_final_audit.select(
    F.count("*").alias("total_transactions"),
    F.countDistinct("transaction_id").alias("unique_transactions"),
    F.sum(
        F.when(
            F.col("transaction_id") == "TRX_LATE_000001",
            1
        ).otherwise(0)
    ).alias("late_transaction_occurrences")
).display()